# O público da Sala São Paulo em números

Dashboard exploratório — 2024 | 12 visões interativas com Plotly

In [11]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# --- Config ---
CAPACIDADE = 1484
REFERENCIA_OCUPACAO = 0.70

# Paleta sofisticada
PAL = {
    'primary': '#1B2838',
    'accent': '#2E86AB',
    'gold': '#E8A838',
    'green': '#56B870',
    'red': '#E05263',
    'light': '#F7F9FC',
    'muted': '#8A9BAE',
    'dark_blue': '#0F4C75',
    'teal': '#3CAEA3',
}
CORES_MODAL = ['#2E86AB', '#E8A838', '#56B870', '#E05263', '#8A5CF5', '#F77F00', '#3CAEA3']

# --- Dados ---
ARQUIVO = Path('dados_sala_sao_paulo_2024.xlsx')
if not ARQUIVO.exists():
    ARQUIVO = Path('/Users/guilherme.nacarelli/.wolf/file-cache/5cc95f6b-a1b1-47f7-9ea4-b42f79ccd7ab-1784337257258-jlru14.xlsx')

base = pd.read_excel(ARQUIVO, sheet_name='02_Base_2024', skiprows=2).dropna(subset=['Modalidade'])
base.columns = ['Ano', 'Modalidade', 'Apresentações', 'Presenças', 'Página PDF', 'Natureza', 'Fonte', 'Média por apresentação', 'Ocupação estimada']
base = base[base['Ano'].astype(str) == '2024'].copy()
base['Apresentações'] = base['Apresentações'].astype(int)
base['Presenças'] = base['Presenças'].astype(int)
base['Média por apresentação'] = base['Presenças'] / base['Apresentações']
base['Ocupação estimada'] = base['Presenças'] / (base['Apresentações'] * CAPACIDADE)
base['Participação'] = base['Presenças'] / base['Presenças'].sum()

# Rótulos curtos
base['Rótulo'] = (base['Modalidade']
    .str.replace('Concertos ', '', regex=False)
    .str.replace(' da Osesp', '', regex=False)
    .str.replace('Populares — ', '', regex=False)
)

# Recortes recentes
recentes = pd.read_excel(ARQUIVO, sheet_name='04_Recortes_recentes', skiprows=2).dropna(subset=['Modalidade'])
recentes.columns = ['Ano', 'Período', 'Modalidade', 'Apresentações', 'Presenças', 'Página PDF', 'Fonte', 'Média por apresentação', 'Ocupação estimada']

comparavel_2024 = base.loc[
    base['Modalidade'].isin(['Concertos sinfônicos da Osesp', 'Ensaios gerais abertos']),
    ['Ano', 'Modalidade', 'Apresentações', 'Presenças'],
].copy()
comparavel_2024['Período'] = 'ano completo'
comparavel_2024['Ano'] = comparavel_2024['Ano'].astype(str)
comparavel_2024['Modalidade'] = comparavel_2024['Modalidade'].replace({'Concertos sinfônicos da Osesp': 'Concertos sinfônicos'})

comparavel = pd.concat([
    comparavel_2024,
    recentes.loc[
        recentes['Modalidade'].isin(['Concertos sinfônicos', 'Ensaios gerais abertos']),
        ['Ano', 'Período', 'Modalidade', 'Apresentações', 'Presenças'],
    ],
], ignore_index=True)
serie = comparavel.groupby(['Ano', 'Período'], as_index=False)[['Apresentações', 'Presenças']].sum()
serie['Média por apresentação'] = serie['Presenças'] / serie['Apresentações']
serie['Ocupação estimada'] = serie['Presenças'] / (serie['Apresentações'] * CAPACIDADE)
serie = serie.sort_values('Ano')
serie['Label'] = serie['Ano'].astype(str) + '<br>' + serie['Período']

total_apresentacoes = int(base['Apresentações'].sum())
total_presencas = int(base['Presenças'].sum())
media_ponderada = total_presencas / total_apresentacoes
ocupacao_total = total_presencas / (total_apresentacoes * CAPACIDADE)

print(f'✓ Dados carregados: {len(base)} modalidades, {total_apresentacoes} apresentações, {total_presencas:,} presenças')

✓ Dados carregados: 7 modalidades, 182 apresentações, 176,933 presenças


## KPIs principais

In [15]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

kpis = [
    ("Apresentações", f"{total_apresentacoes}", "concertos pactuados"),
    ("Presenças", f"{total_presencas:,}".replace(",", "."), "não são pessoas únicas"),
    ("Média<br>ponderada", f"{media_ponderada:,.1f}".replace(",", "."), "presenças por apresentação"),
    ("Ocupação<br>estimada", f"{ocupacao_total:.1%}".replace(".", ","), "capacidade de 1.484 lugares"),
]

fig = make_subplots(
    rows=1,
    cols=4,
    horizontal_spacing=0.06
)

# Remove eixos
for i in range(1, 5):
    fig.update_xaxes(visible=False, row=1, col=i)
    fig.update_yaxes(visible=False, row=1, col=i)

# Adiciona os cards
for i, (titulo, valor, nota) in enumerate(kpis):

    # Limites do card em coordenadas "paper"
    x0 = i / 4 + 0.015
    x1 = (i + 1) / 4 - 0.015
    xc = (x0 + x1) / 2

    # Card branco
    fig.add_shape(
        type="rect",
        xref="paper",
        yref="paper",
        x0=x0,
        x1=x1,
        y0=0.12,
        y1=0.90,
        fillcolor="white",
        line=dict(color="#DDDDDD", width=1),
        layer="below"
    )

    # Valor
    fig.add_annotation(
        x=xc,
        y=0.74,
        xref="paper",
        yref="paper",
        text=f"<b>{valor}</b>",
        showarrow=False,
        font=dict(size=34, color=PAL["primary"]),
        align="center",
        xanchor="center",
        yanchor="middle",
    )

    # Título
    fig.add_annotation(
        x=xc,
        y=0.42,
        xref="paper",
        yref="paper",
        text=f"<b>{titulo}</b>",
        showarrow=False,
        font=dict(size=15, color=PAL["primary"]),
        align="center",
        xanchor="center",
        yanchor="middle",
    )

    # Nota
    fig.add_annotation(
        x=xc,
        y=0.22,
        xref="paper",
        yref="paper",
        text=nota,
        showarrow=False,
        font=dict(size=11, color=PAL["muted"]),
        align="center",
        xanchor="center",
        yanchor="middle",
    )

fig.update_layout(
    width=1600,
    height=280,
    paper_bgcolor=PAL["light"],
    plot_bgcolor=PAL["light"],
    margin=dict(l=30, r=30, t=70, b=20),
    title=dict(
        text="<b>O público da Sala São Paulo em números — 2024</b>",
        x=0.5,
        font=dict(size=22, color=PAL["primary"])
    ),
)

fig.show()

## Comparações por modalidade

In [ ]:
ord_p = base.sort_values('Presenças')
ord_a = base.sort_values('Apresentações')
ord_m = base.sort_values('Média por apresentação')
ord_o = base.sort_values('Ocupação estimada')

fig_bar = make_subplots(rows=1, cols=4, subplot_titles=[
    '<b>Presenças</b>', '<b>Apresentações</b>', '<b>Média por apresentação</b>', '<b>Ocupação estimada</b>'
], horizontal_spacing=0.08)

fig_bar.add_trace(go.Bar(
    y=ord_p['Rótulo'], x=ord_p['Presenças'], orientation='h',
    marker_color=PAL['accent'], text=ord_p['Presenças'].apply(lambda v: f'{v:,.0f}'.replace(',','.')),
    textposition='outside', textfont=dict(size=10),
), row=1, col=1)

fig_bar.add_trace(go.Bar(
    y=ord_a['Rótulo'], x=ord_a['Apresentações'], orientation='h',
    marker_color=PAL['green'], text=ord_a['Apresentações'],
    textposition='outside', textfont=dict(size=10),
), row=1, col=2)

fig_bar.add_trace(go.Bar(
    y=ord_m['Rótulo'], x=ord_m['Média por apresentação'], orientation='h',
    marker_color=PAL['gold'], text=ord_m['Média por apresentação'].apply(lambda v: f'{v:,.0f}'),
    textposition='outside', textfont=dict(size=10),
), row=1, col=3)

fig_bar.add_trace(go.Bar(
    y=ord_o['Rótulo'], x=ord_o['Ocupação estimada'] * 100, orientation='h',
    marker_color=PAL['dark_blue'], text=ord_o['Ocupação estimada'].apply(lambda v: f'{v:.0%}'),
    textposition='outside', textfont=dict(size=10),
), row=1, col=4)

fig_bar.update_layout(
    height=420, showlegend=False,
    margin=dict(l=10, r=30, t=60, b=30),
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(family='Inter, system-ui, sans-serif', size=11, color=PAL['primary']),
)
fig_bar.update_xaxes(showgrid=True, gridcolor='#EEF2F6', zeroline=False)
fig_bar.update_yaxes(showgrid=False)
fig_bar.show()

## Distância da referência de 70% e participação

In [ ]:
# Visão 9: distância da referência
gap = base.copy()
gap['Diferença'] = gap['Ocupação estimada'] - REFERENCIA_OCUPACAO
gap = gap.sort_values('Diferença')

fig_gap = make_subplots(rows=1, cols=2, subplot_titles=['<b>Distância da meta de 70%</b>', '<b>Participação nas presenças</b>'],
                        specs=[[{'type': 'bar'}, {'type': 'pie'}]], horizontal_spacing=0.12)

colors_gap = [PAL['red'] if v < 0 else PAL['green'] for v in gap['Diferença']]
fig_gap.add_trace(go.Bar(
    y=gap['Rótulo'], x=gap['Diferença'] * 100, orientation='h',
    marker_color=colors_gap,
    text=gap['Diferença'].apply(lambda v: f'{v:+.0%}'),
    textposition='outside', textfont=dict(size=10),
    hovertemplate='%{y}: %{x:.1f} pp<extra></extra>',
), row=1, col=1)
fig_gap.add_vline(x=0, line_color=PAL['primary'], line_width=1, row=1, col=1)

# Visão 10: donut
part = base.sort_values('Presenças', ascending=False)
fig_gap.add_trace(go.Pie(
    labels=part['Rótulo'], values=part['Presenças'],
    hole=0.5, marker=dict(colors=CORES_MODAL),
    textinfo='label+percent', textfont=dict(size=10),
    hovertemplate='%{label}<br>%{value:,.0f} presenças<br>%{percent}<extra></extra>',
), row=1, col=2)

fig_gap.update_layout(
    height=400, showlegend=False,
    margin=dict(l=10, r=30, t=60, b=30),
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(family='Inter, system-ui, sans-serif', size=11, color=PAL['primary']),
)
fig_gap.update_xaxes(showgrid=True, gridcolor='#EEF2F6', zeroline=False)
fig_gap.update_yaxes(showgrid=False)
fig_gap.show()

## Evolução — recorte comparável (sinfônicos + ensaios gerais)

In [ ]:
fig_evo = make_subplots(rows=1, cols=2, subplot_titles=[
    '<b>Média por apresentação</b>', '<b>Ocupação estimada</b>'
], horizontal_spacing=0.1)

# Média
fig_evo.add_trace(go.Scatter(
    x=serie['Label'], y=serie['Média por apresentação'],
    mode='lines+markers+text', line=dict(color=PAL['accent'], width=3),
    marker=dict(size=10, color=PAL['accent']),
    text=serie['Média por apresentação'].apply(lambda v: f'{v:.0f}'),
    textposition='top center', textfont=dict(size=11, color=PAL['primary']),
    hovertemplate='%{x}<br>%{y:.0f} presenças/apresentação<extra></extra>',
), row=1, col=1)

# Ocupação
fig_evo.add_trace(go.Scatter(
    x=serie['Label'], y=serie['Ocupação estimada'] * 100,
    mode='lines+markers+text', line=dict(color=PAL['teal'], width=3),
    marker=dict(size=10, color=PAL['teal']),
    text=serie['Ocupação estimada'].apply(lambda v: f'{v:.1%}'),
    textposition='top center', textfont=dict(size=11, color=PAL['primary']),
    hovertemplate='%{x}<br>%{y:.1f}%<extra></extra>',
), row=1, col=2)
fig_evo.add_hline(y=70, line_dash='dash', line_color=PAL['gold'], line_width=2,
                  annotation_text='referência 70%', annotation_font_color=PAL['gold'],
                  row=1, col=2)

fig_evo.update_layout(
    height=350, showlegend=False,
    margin=dict(l=20, r=30, t=60, b=40),
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(family='Inter, system-ui, sans-serif', size=11, color=PAL['primary']),
)
fig_evo.update_xaxes(showgrid=False)
fig_evo.update_yaxes(showgrid=True, gridcolor='#EEF2F6', zeroline=False)
fig_evo.show()

## Limitações

- Presenças ≠ pessoas únicas.
- Ocupação usa capacidade física (1.484); não há dados de assentos ofertados por sessão.
- Dados de 2025/2026 são parciais — comparar apenas por apresentação.
- Pesquisa de satisfação cobre respondentes cadastrados, não todo o público.